# msticpy — prise en main (IT320, pour aller plus loin)

Ce carnet montre deux usages rapides de [msticpy](https://msticpy.readthedocs.io) et de pandas sur les données du dépôt :

1. extraire automatiquement les IoC du rapport acteur fil rouge (`data/rapport-acteur.txt`, acteur fictif SILEX-7) avec `IoCExtract` ;
2. compter les IP sources les plus actives du journal Cowrie synthétique (`data/cowrie/cowrie-24h-sample.json`).

Lancement depuis le conteneur d'outils (msticpy et JupyterLab y sont installés) :

```
docker compose --profile tools run --rm -p 8888:8888 osint-tools jupyter lab --ip 0.0.0.0 --allow-root --no-browser /work/notebooks
```

Les chemins ci-dessous sont relatifs à la racine `plateforme/` (`/work` dans le conteneur). Toutes les données sont fictives (RFC 5737, `.example`).

In [ ]:
from pathlib import Path
import pandas as pd

# Racine du dépôt plateforme : /work dans le conteneur, ou le dossier parent du carnet sur le poste
ROOT = Path('/work') if Path('/work/data').exists() else Path.cwd().parent
RAPPORT = ROOT / 'data' / 'rapport-acteur.txt'
COWRIE = ROOT / 'data' / 'cowrie' / 'cowrie-24h-sample.json'
print(RAPPORT.exists(), COWRIE.exists())

## 1. Extraction d'IoC du rapport avec `IoCExtract`

`IoCExtract` applique des expressions régulières (IPv4, domaines, URL, hash, chemins Windows…). Comparez sa sortie à votre propre
relevé du TP 3 (A1) : que rate-t-il (mutex, clé de registre, User-Agent) ? Que remonte-t-il en trop (références, versions) ?

In [ ]:
from msticpy.transform.iocextract import IoCExtract

texte = RAPPORT.read_text(encoding='utf-8')
extracteur = IoCExtract()
iocs = extracteur.extract(src=texte)   # dict : type -> ensemble de valeurs
for type_ioc, valeurs in sorted(iocs.items()):
    print(f"{type_ioc:12s} {len(valeurs):3d}  ", sorted(valeurs)[:5])

In [ ]:
# Même extraction sous forme de DataFrame, pratique pour filtrer / exporter vers MISP
df_iocs = extracteur.extract_df(texte)
df_iocs[df_iocs['IoCType'].isin(['ipv4', 'dns', 'sha256_hash'])].drop_duplicates('Observable')

## 2. IP sources les plus actives du honeypot Cowrie (pandas)

Le journal est au format « une ligne JSON par événement ». On ne garde que les connexions (`cowrie.session.connect`).

In [ ]:
cowrie = pd.read_json(COWRIE, lines=True)
connexions = cowrie[cowrie['eventid'] == 'cowrie.session.connect']
top = connexions['src_ip'].value_counts().head(10).rename('sessions').to_frame()
top

In [ ]:
# Identifiants les plus tentés (login échoués), pour l'objet MISP « credential » du TP 2
echecs = cowrie[cowrie['eventid'] == 'cowrie.login.failed']
echecs.groupby(['username', 'password']).size().sort_values(ascending=False).head(10)

## Pour aller plus loin

- Enrichir les IP du rapport avec un fournisseur de *threat intelligence* (`msticpy.context.tilookup.TILookup`) : nécessite une clé API
  (OTX, VirusTotal…) dans `msticpyconfig.yaml` — ne jamais la mettre dans le dépôt.
- Croiser `top` avec les attributs `ip-src` déjà présents dans votre MISP (`tools/misp_client.py`, méthode `search_attributes`).